# DBA challenge: live PostgreSQL benchmark

Run this notebook after `nyc311 db-init` and `nyc311 db-load`. It measures the five live dashboard statements using PostgreSQL `EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON)`, excludes one warm-up, records seven server execution times, then applies the candidate index migration and repeats. It also compares insert cost into temporary tables with and without a seller index.

No benchmark values are embedded. The runner writes measured plans, timings and index sizes to `reports/performance_results.json`. Do not interpret results from a different server/data snapshot as comparable.

In [ ]:
import os
assert os.environ.get('DATABASE_URL'), 'Set DATABASE_URL before running this notebook'
from scripts.benchmark_postgres import run
results = run(runs=7)
[(f'Q{q}', results['before_indexes'][q]['median_execution_ms'], results['after_indexes'][q]['median_execution_ms']) for q in results['before_indexes']]

## Reading results

For each query, compare the median, min/max, planning time, estimated versus actual rows, shared buffer reads/hits, and full JSON plan in the result artifact. The performance budget is 200 ms per query. If an index does not appear in a plan or fails to improve measurements, record that as evidence; do not report a speedup based only on the existence of an index. The script removes the low-selectivity customer-state candidate after its evaluation and records its removal.

Benchmark artifact: `reports/performance_results.json`. The runner leaves the useful indexes in place.